# Color spaces

Convert between OKLab, OKLCh, HSV, Display P3, Rec.2100 and more with lcms2.spaces, mixed with ICC profiles.

In [1]:
import numpy as np

import lcms2
from lcms2 import spaces

`lcms2.spaces` has color spaces defined as NumPy functions. `ColorSpace.list()` shows the registered
names; `spaces.get(name)` returns one, and the ready-made instances are module attributes.

In [2]:
print(spaces.ColorSpace.list())
print(spaces.get("oklab"), spaces.oklab.n_channels, spaces.oklab.device_domain)

['a98-rgb', 'cmyk', 'display-p3', 'display-p3-linear', 'gray', 'hsv', 'hwb', 'lch', 'loggray', 'oklab', 'oklch', 'prophoto-rgb', 'rec2020', 'rec2100-hlg', 'rec2100-linear', 'rec2100-pq', 'srgb', 'srgb-linear', 'xyz-d50', 'xyz-d65']
<OKLabSpace name='oklab' n_channels=3> 3 [(0.0, 1.0), (-0.4, 0.4), (-0.4, 0.4)]


`space.to(target, values)` converts, with the target a name or a space. Values are shaped
`(..., n_channels)` and in the space's own range. Hue channels (HSV, HWB, LCh, OKLCh) are a fraction of a turn.

In [3]:
orange = np.array([1.0, 0.5, 0.0])
for name in ("hsv", "hwb", "oklab", "oklch", "display-p3", "rec2100-pq", "xyz-d65"):
    print(f"{name:11s}", spaces.srgb.to(name, orange).round(4))

hsv         [0.0833 1.     1.    ]
hwb         [0.0833 0.     0.    ]
oklab       [0.7311 0.1126 0.1482]
oklch       [0.7311 0.1861 0.1466]
display-p3  [0.936  0.5273 0.1983]
rec2100-pq  [0.9621 0.8584 0.639 ]
xyz-d65     [0.489  0.3657 0.0448]


Spaces round-trip, and every conversion goes through XYZ D50, available as `to_xyz` and `from_xyz`.

In [4]:
hsv = spaces.srgb.to("hsv", orange)
print(spaces.hsv.to("srgb", hsv).round(6))
xyz = spaces.srgb.to_xyz(orange)
print(xyz.round(4), spaces.srgb.from_xyz(xyz).round(6))

[1.  0.5 0. ]
[0.5185 0.376  0.0347] [1.  0.5 0. ]


Spaces and ICC profiles mix freely in `lcms2.convert` and in `ColorSpace.to`. When a space is involved
the conversion runs through XYZ in float64, so intents do not apply.

In [5]:
print(lcms2.convert([0.7, 0.1, 0.05], "oklab", "Display P3").round(4))
print(spaces.oklch.to("Adobe RGB (1998)", [0.7, 0.15, 0.3]).round(4))
print(lcms2.convert(np.array([200, 100, 50], np.uint8), "sRGB", "oklch").round(4))

[0.8119 0.5268 0.4868]
[0.6492 0.6382 0.1559]
[0.6138 0.1423 0.1252]


Space and profile link up: `space.profile` is a cached ICC profile of the space, and `profile.space`
is a space that evaluates a profile.

In [6]:
print(spaces.oklab.profile)
print(lcms2.Profile("Display P3").space)

<Profile 'OKLabSpace (python-lcms2)' 3CLR colorspace v4.4>
<ICCSpace name='Display P3' n_channels=3>


`RGBSpace` is the shortcut for RGB spaces: set a `matrix` (linear RGB to XYZ D50) and optional
`decode`/`encode` curves. Subclasses register themselves under their name.

In [7]:
class Linear709(spaces.RGBSpace):
    matrix = spaces.srgb.matrix


print(Linear709().to("srgb", [0.5, 0.5, 0.5]).round(4))
print("linear709" in spaces.ColorSpace.list(), [n for n in spaces.ColorSpace.list() if "709" in n])

[0.7354 0.7354 0.7354]
True ['linear709']
